# Robotics III (Lecture 04)

## 第一部分：运动规划 (Motion Planning)

运动规划的核心任务是在存在障碍物的环境中，寻找一条从起始状态到目标状态的无碰撞几何路径。

### 1. 问题定义 (Problem Formulation)
*   **配置空间 (Configuration Space, $\mathcal{C}\text{-space}$)**:
    *   定义：包含系统所有可能状态的集合，通常是 $\mathbb{R}^n$ 的子集。
    *   $\mathcal{C}_{free}$：所有无碰撞的合法状态集合。
    *   $\mathcal{C}_{obs}$：障碍物占据的状态集合。
    *   关系：$\mathcal{C} = \mathcal{C}_{free} \cup \mathcal{C}_{obs}$。
*   **规划目标**:
    *   给定 $\mathcal{C}_{free}$，起始状态 $q_{start}$ 和目标状态 $q_{goal}$。
    *   计算一系列连续动作（路径），使得机器人从 $start$ 移动到 $goal$ 且全程位于 $\mathcal{C}_{free}$ 中。

### 2. 碰撞检测建模 (Collision Modeling)
为了判断一个状态 $q$ 是否属于 $\mathcal{C}_{free}$，必须进行碰撞检测。
*   **模型简化**:
    *   **Visual Mesh**: 用于渲染，顶点多，计算昂贵。如Triangle Mesh（三角面片网格）。
    *   **Collision Mesh**: 用于物理计算，通常比 Visual Mesh 简单。如简化的球体组合。
*   **凸分解 (Convex Decomposition)**:
    *   **原因**: 凸多边形/凸多面体的碰撞检测远比非凸物体高效。
    *   **方法**:
        *   *Convex-Hull*: 单一凸包，效率最高但精度最低。
        *   *Exact Convex Decomposition*: NP-hard，生成过多碎片，不实用。
        *   *Approximate Convex Decomposition (ACD)*: 实用方案。在保证凹陷度（concavity）低于阈值的前提下，将网格分割为最少数量的凸块。

### 3. 基于采样的算法 (Sampling-based Algorithms)
不精确构建整个 $\mathcal{C}\text{-space}$，而是通过随机采样探索空间。
*   **优点**: 概率完备性 (Probabilistically complete)，适用于高维空间。
*   **缺点**: 无法保证最优性，在狭窄通道 (Narrow Passages) 表现不佳。

#### A. 概率路图法 (Probabilistic Roadmap, PRM)
适用于**多查询 (Multi-query)** 和静态场景。
1.  **构图阶段 (Map Construction)**:
    *   在 $\mathcal{C}_{free}$ 中随机采样点。
    *   将采样点与其邻域内的点连接（需检测边是否碰撞）。
2.  **查询阶段 (Query)**:
    *   将 $q_{start}$ 和 $q_{goal}$ 接入图中。
    *   使用 Dijkstra 或 A* 算法搜索路径。
*   **采样优化策略**:
    *   *Uniform Sampling*: 均匀采样。
    *   *Gaussian Sampling*: 在障碍物边缘采样（高斯分布扰动），解决紧贴障碍物的路径问题。
    *   *Bridge Sampling*: 专门针对狭窄通道（Narrow Bridge），通过检测“障碍-空闲-障碍”模式来采样桥梁中点。

#### B. 快速扩展随机树 (Rapidly-exploring Random Trees, RRT)
适用于**单次查询 (Single-query)** 场景。
*   **核心逻辑**:
    1.  从 $q_{start}$ 开始生长一棵树 $\mathcal{T}$。
    2.  **采样**: 随机生成目标点 $q_{target}$（以概率 $\beta$ 选 $q_{goal}$，否则在 $\mathcal{C}_{free}$ 随机选）。
    3.  **最近邻**: 在树中找到离 $q_{target}$ 最近的节点 $q_{near}$。
    4.  **扩展 (Extend)**: 从 $q_{near}$ 向 $q_{target}$ 移动步长 $\epsilon$，生成新节点 $q_{new}$。
        $$ q_{new} \leftarrow q_{near} + \frac{\epsilon}{|q_{target} - q_{near}|}(q_{target} - q_{near}) $$
    5.  **检测**: 若 $q_{new}$ 及路径无碰撞，将 $q_{new}$ 加入树。
*   **RRT-Connect**:
    *   双向生长：从 $start$ 和 $goal$ 同时生长两棵树。
    *   贪心策略：一棵树试图直接连接到另一棵树的新节点，大大加速收敛。

### 4. 路径后处理 (Shortcutting)
采样算法生成的路径通常是曲折、不自然的（jerky），且非最优。
*   **算法**:
    1.  在路径上随机取两点 $u, v$。
    2.  检测 $u, v$ 之间的直线是否无碰撞 ($Visible(u, v)$)。
    3.  若无碰撞，用直线代替原有的曲折路径。
    4.  重复迭代。

### 5. 工具库
*   **OMPL (Open Motion Planning Library)**: ROS MoveIt 的默认规划库，包含 RRT, RRT-Connect, PRM 等几何规划器。